# 面试问题：Prompt、RAG、微调和 Tool Calling 应该怎样选择？

可直接复述的回答：先判断缺口属于知识、行为还是外部动作，再选择最小充分方案。稳定且简单的表达约束优先 Prompt；知识频繁变化、需要引用时使用 RAG；跨大量样本反复出现的稳定行为模式才考虑微调；任何读取账户或改变世界状态的操作必须走 Tool Calling。真实系统通常是组合架构，而不是四选一。比较方案时要在同一黄金集上同时看质量、成本、延迟和安全失败。上线还要绑定知识版本、模型版本、工具权限与回滚策略。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：电商客服请求与输入预览

数据来自脱敏后的客服意图结构，保留了问题文本、知识时效、是否需要账户动作、行为是否稳定和人工标注路线。六条记录覆盖政策问答、文案格式、订单动作和组合问题；它们是离线教学样本，不能外推线上分布。


In [1]:
cases01 = [  # 构造带真实决策字段的客服样本。
    {"id": "q1", "question": "今天北京仓还能次日达吗", "volatility": 2, "needs_action": False, "stable_behavior": False, "gold": "rag"},  # 时效库存知识需要检索。
    {"id": "q2", "question": "把回复改成三点式并保持礼貌", "volatility": 0, "needs_action": False, "stable_behavior": False, "gold": "prompt"},  # 单次表达约束适合提示词。
    {"id": "q3", "question": "为我的订单申请退款", "volatility": 1, "needs_action": True, "stable_behavior": False, "gold": "tool"},  # 改变订单状态必须调用工具。
    {"id": "q4", "question": "每天将商品标题改写成平台风格", "volatility": 0, "needs_action": False, "stable_behavior": True, "gold": "finetune"},  # 高频稳定行为适合微调。
    {"id": "q5", "question": "查询最新会员规则并解释积分", "volatility": 2, "needs_action": False, "stable_behavior": False, "gold": "rag"},  # 新规则需要版本化证据。
    {"id": "q6", "question": "按最新政策检查订单并补发", "volatility": 2, "needs_action": True, "stable_behavior": False, "gold": "tool"},  # 检索知识后仍需受控动作。
]  # 完成六条可读业务记录。
print("教学实验输入：客服请求决策字段")  # 标明当前输出是输入预览。
for case01 in cases01:  # 逐条展示原始业务记录。
    print(case01)  # 输出问题与人工路线供读者核对。


教学实验输入：客服请求决策字段
{'id': 'q1', 'question': '今天北京仓还能次日达吗', 'volatility': 2, 'needs_action': False, 'stable_behavior': False, 'gold': 'rag'}
{'id': 'q2', 'question': '把回复改成三点式并保持礼貌', 'volatility': 0, 'needs_action': False, 'stable_behavior': False, 'gold': 'prompt'}
{'id': 'q3', 'question': '为我的订单申请退款', 'volatility': 1, 'needs_action': True, 'stable_behavior': False, 'gold': 'tool'}
{'id': 'q4', 'question': '每天将商品标题改写成平台风格', 'volatility': 0, 'needs_action': False, 'stable_behavior': True, 'gold': 'finetune'}
{'id': 'q5', 'question': '查询最新会员规则并解释积分', 'volatility': 2, 'needs_action': False, 'stable_behavior': False, 'gold': 'rag'}
{'id': 'q6', 'question': '按最新政策检查订单并补发', 'volatility': 2, 'needs_action': True, 'stable_behavior': False, 'gold': 'tool'}


## 2. Baseline（基线）：所有请求只写 Prompt

最便宜的基线是假设提示词能够解决全部问题。它能处理格式要求，却无法获得当天库存，也不能安全修改订单。先展示这个基线，避免后面只说组合策略“更好”。


In [2]:
def baseline_route01(case01):  # 定义只依赖提示词的朴素路线。
    return "prompt"  # 对所有请求返回同一个方案。
baseline_rows01 = []  # 收集逐样本基线结果。
for case01 in cases01:  # 在同一批客服请求上运行基线。
    prediction01 = baseline_route01(case01)  # 获取基线预测路线。
    baseline_rows01.append((case01["id"], prediction01, case01["gold"], prediction01 == case01["gold"]))  # 保存预测、真值和命中标记。
baseline_accuracy01 = sum(row01[3] for row01 in baseline_rows01) / len(baseline_rows01)  # 计算基线准确率。
print("基线路线表：id | 预测 | 人工路线 | 是否正确")  # 输出结果表表头。
for row01 in baseline_rows01:  # 逐行展示基线错误分布。
    print(row01)  # 输出一条请求的路线对照。
print("基线准确率", round(baseline_accuracy01, 3))  # 展示同一指标下的基线表现。


基线路线表：id | 预测 | 人工路线 | 是否正确
('q1', 'prompt', 'rag', False)
('q2', 'prompt', 'prompt', True)
('q3', 'prompt', 'tool', False)
('q4', 'prompt', 'finetune', False)
('q5', 'prompt', 'rag', False)
('q6', 'prompt', 'tool', False)
基线准确率 0.167


## 3. 核心实现：知识、行为与动作缺口评分

动作缺口拥有最高优先级，因为仅生成文字不能完成订单变更。知识时效高时选择带版本和引用的 RAG；稳定且重复的行为缺口才进入微调；剩余简单表达任务留给 Prompt。输出每条请求的四路分数，展示决策过程而非只给标签。


In [3]:
def route_scores01(case01):  # 计算四种方案的可解释分数。
    scores01 = {"prompt": 1.0, "rag": 0.0, "finetune": 0.0, "tool": 0.0}  # 初始化各路线基础分。
    scores01["rag"] = float(case01["volatility"] * 2)  # 知识越易变越需要检索。
    scores01["finetune"] = 3.0 if case01["stable_behavior"] else 0.0  # 稳定重复行为提高微调分数。
    scores01["tool"] = 5.0 if case01["needs_action"] else 0.0  # 外部动作赋予工具最高优先级。
    return scores01  # 返回可审计的路线分数。
policy_rows01 = []  # 收集策略路线与评分轨迹。
for case01 in cases01:  # 对每条请求运行缺口判断。
    scores01 = route_scores01(case01)  # 获取当前请求的四路分数。
    route01 = max(scores01, key=scores01.get)  # 选择分数最高的最小充分方案。
    policy_rows01.append((case01["id"], scores01, route01, case01["gold"]))  # 保存过程分数和最终路线。
print("核心过程：id | 四路分数 | 选择 | 人工路线")  # 输出评分轨迹表头。
for row01 in policy_rows01:  # 逐条展示可解释决策。
    print(row01)  # 输出当前请求的全部中间分数。


核心过程：id | 四路分数 | 选择 | 人工路线
('q1', {'prompt': 1.0, 'rag': 4.0, 'finetune': 0.0, 'tool': 0.0}, 'rag', 'rag')
('q2', {'prompt': 1.0, 'rag': 0.0, 'finetune': 0.0, 'tool': 0.0}, 'prompt', 'prompt')
('q3', {'prompt': 1.0, 'rag': 2.0, 'finetune': 0.0, 'tool': 5.0}, 'tool', 'tool')
('q4', {'prompt': 1.0, 'rag': 0.0, 'finetune': 3.0, 'tool': 0.0}, 'finetune', 'finetune')
('q5', {'prompt': 1.0, 'rag': 4.0, 'finetune': 0.0, 'tool': 0.0}, 'rag', 'rag')
('q6', {'prompt': 1.0, 'rag': 4.0, 'finetune': 0.0, 'tool': 5.0}, 'tool', 'tool')


## 4. 结果表与结果解读

组合策略在这组六条教学样本上修复了知识和动作缺口。结果不说明线上一定达到相同准确率，只说明决策规则与人工路线一致；真正评估还要按意图切片，并测量回答质量、引用正确率、工具成功率、成本和 P95 延迟。


In [4]:
policy_predictions01 = {row01[0]: row01[2] for row01 in policy_rows01}  # 建立请求到策略路线的映射。
policy_accuracy01 = sum(policy_predictions01[case01["id"]] == case01["gold"] for case01 in cases01) / len(cases01)  # 计算策略准确率。
cost01 = {"prompt": 1.0, "rag": 2.2, "finetune": 1.4, "tool": 2.8}  # 设置教学用相对成本。
average_cost01 = sum(cost01[policy_predictions01[case01["id"]]] for case01 in cases01) / len(cases01)  # 汇总组合策略平均成本。
print("方案 | 路线准确率 | 教学相对成本")  # 输出可比较结果表头。
print("全 Prompt", round(baseline_accuracy01, 3), 1.0)  # 展示朴素方案结果。
print("缺口路由", round(policy_accuracy01, 3), round(average_cost01, 3))  # 展示核心策略结果。
print("结果解读：更高成本来自检索和工具，但换回了时效知识与真实动作能力")  # 解释质量成本权衡。


方案 | 路线准确率 | 教学相对成本
全 Prompt 0.167 1.0
缺口路由 1.0 2.067
结果解读：更高成本来自检索和工具，但换回了时效知识与真实动作能力


## 5. 失败案例与修正：政策问题掩盖了真实动作

只做关键词路由时，“最新政策”会把组合请求错误送到 RAG，模型能够解释政策却没有补发订单。修正方法是先识别副作用和授权需求，再把 RAG 作为工具调用前的知识步骤，而不是让知识路线覆盖动作路线。


In [5]:
failure_case01 = cases01[-1]  # 选取同时包含最新政策与补发动作的请求。
naive_keyword_route01 = "rag" if "政策" in failure_case01["question"] else "prompt"  # 演示关键词优先造成的错误路由。
fixed_route01 = policy_predictions01[failure_case01["id"]]  # 读取动作优先后的修正路线。
composition01 = ["rag:读取政策版本", "tool:校验订单与审批", "tool:执行补发"]  # 给出真实组合执行步骤。
print("失败行为", failure_case01["question"], "->", naive_keyword_route01)  # 展示只解释不执行的错误路线。
print("修正行为", failure_case01["question"], "->", fixed_route01)  # 展示动作优先后的路线。
print("修正后的组合轨迹", composition01)  # 输出知识与动作的完整轨迹。


失败行为 按最新政策检查订单并补发 -> rag
修正行为 按最新政策检查订单并补发 -> tool
修正后的组合轨迹 ['rag:读取政策版本', 'tool:校验订单与审批', 'tool:执行补发']


## 6. 生产边界与发布合同

生产系统需要经过意图模型或规则服务得到缺口字段，RAG 要绑定索引版本和 ACL，工具要绑定主体、审批和幂等键，微调要绑定基础模型与训练数据版本。这里的相对成本和六条样本只是教学值，不能用于容量规划。


In [6]:
release_contract01 = {"router_version": "gap-router-v1", "knowledge_index": "policy-2026-07", "tool_policy": "order-action-v3", "fallback": "human-review"}  # 定义上线必须绑定的版本合同。
print("生产发布合同", release_contract01)  # 展示可审计的路由配置。
print("生产替换点：意图特征服务、版本化检索、授权工具网关、分意图黄金集")  # 说明教学实现与线上组件的差距。


生产发布合同 {'router_version': 'gap-router-v1', 'knowledge_index': 'policy-2026-07', 'tool_policy': 'order-action-v3', 'fallback': 'human-review'}
生产替换点：意图特征服务、版本化检索、授权工具网关、分意图黄金集


## 7. 最小回归测试

断言只保护最重要的不变量，不再承担教学输出。


In [7]:
assert len(cases01) >= 5  # 保证案例仍包含足够多的业务样本。
assert policy_accuracy01 > baseline_accuracy01  # 保证核心策略优于全提示词基线。
assert policy_predictions01["q3"] == "tool"  # 保证有副作用的退款请求必须走工具。
assert fixed_route01 == "tool"  # 保证组合请求不会被政策关键词错误覆盖。
print("最小回归测试通过：路线、动作优先级与案例规模保持稳定")  # 向学习者显示回归检查已经完成。


最小回归测试通过：路线、动作优先级与案例规模保持稳定
